# Contextualized self-distillation — best config + what it does

**Idea.** The model is its own teacher. Given a privileged **context** (a correct solution found by
best-of-N), it produces a confident full-logit target over `y* = correct CoT + answer`. The **student**
(same net, no context) is trained to match it: `loss = KL(teacher ‖ student)` over `y*`. This internalizes
the model's own successful reasoning → **greedy@1 climbs toward pass@N**.

**Headline result (3 seeds):**

| model | base greedy@1 | pass@8 | **distilled** | Δ |
|---|---|---|---|---|
| Qwen3-0.6B | 0.557 | 0.837 | **0.653** | +9.6 |
| Qwen3-1.7B | 0.710 | 0.898 | **0.816** | +10.6 |

It reliably converts ~⅓–½ of the pass@1→pass@8 gap. Runs on **cuda:1**.

$\text{@FY}$. I believe this is fairly close to improvement obtainable from GRPO. 

### Comment @FY
- (Obs 1). Looks like bigger model have a smaller gap between greedy@1 and pass@8? This is confonded by the fact that bigger model nearly saturates GSM8K anyway. 

## 0. Setup — student (trainable) + FROZEN teacher
Two non-negotiables (both cause silent failure):
1. **Frozen teacher.** Sharing live weights with the student makes the target drift → collapse to 0%.
2. **`weight_decay=0`.** AdamW's default 0.01 decays weights even at zero gradient → silent degradation.

The teacher can be a snapshot of the student (self-distill, 0.653) **or a moderately stronger same-tokenizer
model** (`Qwen3-1.7B` → 0.6B was the single best: 0.660). A *too*-strong teacher (8B→0.6B) **hurts** — the
capacity gap (see §6).

$\text{@FY}$. This is an very interesting findings, contextual distillation HURT under off-policy $y^{*}$. 1.7B helps but 8B hurts are very interesting result --- I wonder what's the source or the "capacity gap"? that the logits bigger model hold are 
simply too informative for the smaller model to predict? If we SVD those logits, I wonder whether we can extract some interesting insight (e.g. if we drop the tails in singular directions from big model, will the distillation be easier? where exactly is the capacity gap located?) 

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import sys, json, random
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJ = Path.cwd().parent if Path.cwd().name == "notebook" else Path("/home/claudeuser/arl")
sys.path.insert(0, str(PROJ))
from script.context_self_distill import build_pair, kl_loss_one, quick_eval   # validated helpers
from script.grpo_gsm8k import load_gsm8k

STUDENT = "Qwen/Qwen3-0.6B"
TEACHER = "Qwen/Qwen3-0.6B"          # self-distill (0.653). Try "Qwen/Qwen3-1.7B" for the best 0.6B run (0.660)
tok = AutoTokenizer.from_pretrained(STUDENT)
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.bfloat16, attn_implementation="eager").to("cuda").train()
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, dtype=torch.bfloat16, attn_implementation="eager").to("cuda").eval()
for p in teacher.parameters():
    p.requires_grad_(False)          # rule 1: frozen teacher
print("student trainable | teacher frozen:", TEACHER)

## 1. Data — the model's OWN best-of-N correct rollouts (Phase A)

Each row is a verifier-correct `(question, cot, answer, full)`. **Use the model's own rollouts, not GT** —
we tested GSM8K ground-truth CoT (100% coverage) and it did *worse* (0.61 vs 0.65): GT is a foreign style
the small model can't absorb. The model's own correct CoTs are the most learnable target.

Build with `script/gen_best_of_n.py` (use `--keep_k 4` for diverse targets, see §5).

In [ ]:
# python script/gen_best_of_n.py --model Qwen/Qwen3-0.6B --n 8 --limit 3000   (--keep_k 4 for diversity)
data = [json.loads(l) for l in (PROJ / "data/best_of_n/correct.jsonl").open()]
random.Random(0).shuffle(data)
ex = data[0]
print(f"{len(data)} correct rollouts | answer={ex['answer']} | y* tail: {ex['full'][-50:]!r}")

## 2. The gadget — a contextualized teacher/student pair

`build_pair` makes two sequences over the **same** target `y*`: the **teacher** prompt carries the correct
solution as context (→ confident, correct logits); the **student** prompt is the plain question.
`kl_loss_one` reads both models' logits over `y*` and returns `KL(teacher ‖ student)` (teacher detached).

The KL scales with how much the context tells the teacher — `none` (control) must be ~0:

$\text{@FY}$. Did we actually end up using CE loss to mix up with KL loss? If not we should clean up the script to get rid of it. 

In [ ]:
for ctx in ["none", "answer", "cot", "full_cot"]:
    kl = kl_loss_one(student, teacher, tok, ex, context=ctx, span="completion", direction="forward", device="cuda")
    print(f"context={ctx:<9} KL = {kl.item():.4f}")

In [1]:
# Question 1. when you say after contextualized distillation, model improves its accuracy, my question is on the training set
#             where model used to can't answer but now it can, is the model's (correct) answer exactly the one we put into the contextualized distillation training pipeline? 
# [Idea 2].   current contextualized distillation pipeline ONLY distills a fixed set of offline correct responses from the frozen model right? 
#             since we've found on-policy data to be more effective, shouldn't we be trying to collect on-policy best of N rollout correct responses on each (training) queries? 
#             one benefits here, is on-policiness, of course we can maintain a buffer, so that our set of correct rollouts for each query will keep growing (if the model can produce 
#             diverse correct responses), making our contextualized self-distillation pipeline much more robust (I expect us to use a vllm server for quick generation)

# Question 2. how did the "increasing diversity of HINT / answer / CoT" experiment go? did it improve performance at all? 
# -> yes it does, from K (avg. distinct correct for each query) 1 -> 3, we see a 1.6pp improvement in acc (0.63 -> 0.64) with 
#    lower variances across seed
# Question 3. what if we do curriculumn: interpolate plain -> full for hint in teacher context, or interpolate full -> plan in student context? 
#           -> this is an open idea, we need to test it (look into literature for relavant trick about how to "gradually truncate full sequence" or other related tricks, merely pop out more tokens on the prefix / suffix feels unprincipled)

# Question 4. i am concerned about the chat template on the "student full" and "teacher full" token ids
#             i don't see '<think>' tag being applied, but as long as the "full" is extracted with the template structure
#             it's fine, so this is sth I'd need a kernel output to confirm.
#            -> we simply need a notebook kernel output to confirm this

# Request 1. we don't need ce_weight in practice anyway. 
# Request 2. prep a kernel for me to see the printed (non-tokenized t_full, s_full), as well as t_full[t_start:], s_full[s_start:]
#            just to double confirm things

# Question 5. Have we ablated on tau? how should i intuitively understand the effect of 'tau' here? 
#           -> [AI]. Yep 0.65 (tau=1), 0.64 (tau=2), 0.53 (tau=4)
#           -> [FY] ok, so how about tau < 1.0? we will then further sharpen distribution no? have you tried that? 


## 3. Train — the BEST config

`context=full_cot`, `span=completion`, `direction=forward`, `tau=1`, no CE, `weight_decay=0`, lr 1e-5,
~300 steps. We swept all of these (§4) — this is the winner; the others are neutral-or-worse.

In [ ]:
CONTEXT, SPAN, DIRECTION, STEPS, MICRO_BS = "full_cot", "completion", "forward", 300, 2
opt = torch.optim.AdamW(student.parameters(), lr=1e-5, weight_decay=0.0)   # rule 2: wd=0

di = 0
for step in range(1, STEPS + 1):
    opt.zero_grad(); terms = []
    for _ in range(MICRO_BS):
        kl = kl_loss_one(student, teacher, tok, data[di % len(data)], CONTEXT, SPAN, DIRECTION, "cuda")
        di += 1
        if kl is not None:
            terms.append(kl)
    if terms:
        loss = torch.stack(terms).mean(); loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0); opt.step()
    if step % 50 == 0:
        print(f"step {step}/{STEPS}  meanKL={loss.item():.4f}")

In [ ]:
_, test = load_gsm8k()
acc = quick_eval(student, tok, test, n=300)
print(f"greedy@1 after distill = {acc:.4f}   (0.6B base 0.557, self-distill 0.653, pass@8 0.837)")

## 4. What we swept — and what was NEUTRAL (so you don't re-tune it)

All on 0.6B, eval=300, 3 seeds. Baseline = best-of-N forward **0.653 ± 0.012**.

| knob | result | verdict |
|---|---|---|
| more steps (300→2700) | ↓ | undertraining **refuted** (foreign-target overfit) |
| KL direction (reverse) | 0.639 | neutral/worse |
| target = GT CoT (100% coverage) | 0.61–0.62 | **worse** → coverage is *not* the wall |
| temperature τ=2 / τ=4 | 0.641 / 0.531 | worse → keep the **sharp** target |
| CE-mix 0.5 / 1.0 | 0.656 / 0.651 | neutral |

**Takeaway:** the within-capacity knobs are exhausted at ~0.65 for 0.6B. The bottleneck is the student's
**absorption capacity**, not the target or the loss shape.

## 5. The two levers that DO move it (small)

**Diversity** — keep K distinct correct CoTs/question (`gen_best_of_n.py --keep_k 4`) and train on all of
them. Modest but real and more stable:

| targets | greedy@1 |
|---|---|
| single (1 CoT/Q) | 0.632 ± 0.013 |
| diverse (3.4 CoT/Q) | **0.647 ± 0.005** (+1.5) |

**Stronger teacher** — set `TEACHER="Qwen/Qwen3-1.7B"` in §0 (cross-model KL is exact: same vocab).
Run with `script/run_xkd.sh`.

## 6. Teacher scaling — non-monotonic (the capacity gap)

Student 0.6B, same target data, only the teacher changes:

| teacher | distance | greedy@1 |
|---|---|---|
| 0.6B (control) | 1× | 0.627 |
| **1.7B** | 3× | **0.660** ← best |
| 8B | 13× | 0.593 ← *worse* |

**Intuition:** distillation needs the student to be able to *match* the teacher's distribution. 1.7B is
close enough to absorb; 8B is so far that its distribution is unrepresentable for the 0.6B → matching it
degrades it (Mirzadeh TAKD 2020; Cho & Hariharan 2019). To use a big teacher, bridge with a teacher-assistant
chain (8B→1.7B→0.6B).

## TL;DR for picking the next step

- **Method works**: +9.6 (0.6B) / +10.6 (1.7B), confirmed at 1314 (0.6B 0.538→0.644 ± 0.001).
- **0.6B is capacity-capped ~0.65–0.67**; no cheap knob breaks it (coverage, KL dir, τ, CE all flat).
- **Only real levers**: diversity (+1.5) and a *moderately* stronger teacher (1.7B, +0.7; 8B hurts).
- **Biggest practical lever is the student size**: 1.7B self-distills to **0.816** — far above any 0.6B recipe.

Candidate next steps: (a) TA-chain 8B→1.7B→0.6B, (b) stack diversity + 1.7B-teacher and confirm at 1314,
(c) push the 1.7B student (most headroom). Scripts: `run_exp5.sh`, `run_div.sh`, `run_xkd.sh`, `run_xkd8.sh`.